## 1. Setup & Imports

Import all functions from the anomaly detection module.

In [ ]:
# Import all functions from the anomaly detection module
from anomaly_detection_main import (
    # Core detection functions
    run_vulnerability_detection,
    detect_statistical_anomalies,
    detect_dbscan_anomalies,
    
    # Exploration & visualization
    explore_commit_features,
    visualize_isolation_forest_decision_path,
    
    # Result printing (modular)
    print_all_results,
    print_cve_comparison_table,
    print_overall_comparison,
    print_feature_contributions,
    
    # Utilities
    parse_git_log_file,
    get_parsed_log_cached,
    _last_model_info,
)

print("All functions imported successfully!")

---
## 2. Exploratory Visualization (Pandas Repo)

Explore commit features distribution using the pandas repository as an example.

In [ ]:
# Explore commit features in the pandas repository
# This generates visualizations of feature distributions

exploration_result = explore_commit_features(
    repo_path="./git_repos/pandas",
    repo_url="https://github.com/pandas-dev/pandas.git",
    features=[
        'hour_of_day', 
        'day_of_week', 
        'churn_ratio', 
        'msg_length', 
        'lines_inserted', 
        'lines_deleted', 
        'files_changed'
    ],
    n_commits=10000
)

print(f"\nAnalyzed {len(exploration_result['df'])} commits from pandas repo")

---
## 3. XZ Repository Analysis

Analyze the XZ repository (famous for the XZ backdoor CVE-2024-3094) and visualize the Isolation Forest decision path.

In [ ]:
# Run anomaly detection on XZ repository
# This analyzes the repository containing the famous XZ backdoor

xz_results = run_vulnerability_detection(
    specific_repo_path="./git_repos/xz",
    specific_repo_url="https://github.com/tukaani-project/xz.git",
    specific_commit='cf44e4b',  # Known vulnerability-introducing commit
    n_commits=500,
    contamination=0.15,
    use_codebert=False,
    visualize=True,
    print_results=False  # We'll print separately
)

print("\nXZ repository analysis complete!")

In [ ]:
# Visualize the Isolation Forest decision tree (up to 4 levels)
# This shows how the model isolates anomalies

from anomaly_detection_main import _last_model_info

if _last_model_info:
    visualize_isolation_forest_decision_path(
        model=_last_model_info['model'],
        X=_last_model_info['X_scaled'],
        feature_names=_last_model_info['features'],
        sample_idx=0,  # Index of the target commit
        max_depth=4    # Visualize up to 4 levels deep
    )
else:
    print("No model info available. Run the XZ analysis cell first.")

---
## 4. Full Benchmark

Run the complete vulnerability detection benchmark on all cached CVEs.

### 4.1 Run Benchmark Over All Cached CVEs

This runs the anomaly detection algorithms on all vulnerabilities with cached data.

In [ ]:
# Run the full benchmark on all cached CVEs
# Set print_results=False to print each section separately below

benchmark_results = run_vulnerability_detection(
    n_samples=None,           # Analyze all available vulnerabilities
    n_commits=2000,           # Use 2000 commits as context window
    contamination=0.15,       # Primary anomaly threshold (15%)
    cached_only=True,         # Only use repos with fully cached data (fastest)
    local_repos_only=False,   # Fallback: use already-cloned repos
    use_codebert=False,       # Disable CodeBERT for faster execution
    multi_threshold=False,    # Single threshold evaluation
    visualize=False,          # No per-CVE visualizations
    print_results=False       # Don't print results yet - we'll do it section by section
)

print("\n" + "="*80)
print("BENCHMARK COMPLETE!")
print("="*80)
print(f"\nAnalyzed {len(benchmark_results['cve_results'])} CVEs")
print("\nResults stored in 'benchmark_results' variable.")
print("Run the cells below to see detailed breakdowns.")

### 4.2 Feature Contributions Per Method

Shows which features contribute most to anomaly detection for each method.

In [ ]:
# Print feature importance/contributions for each detection method

print_feature_contributions(benchmark_results)

### 4.3 CVE-wise Comparison Table

Detailed breakdown showing detection results for each CVE across all methods.

In [ ]:
# Print the CVE-wise comparison table

print_cve_comparison_table(benchmark_results)

### 4.4 Overall Comparison Metrics

Aggregate performance metrics comparing all detection methods.

In [ ]:
# Print overall/aggregate comparison metrics

print_overall_comparison(benchmark_results)

---
## Summary

Quick summary of results:

In [ ]:
# Print a quick summary of the benchmark results

summary = benchmark_results['summary']

print("="*60)
print("  DETECTION SUMMARY")
print("="*60)

for method, stats in summary.items():
    if stats['total'] > 0:
        recall = stats['detected'] / stats['total'] * 100
        print(f"\n{method}:")
        print(f"  Detected: {stats['detected']}/{stats['total']} ({recall:.1f}%)")
        print(f"  Mean Percentile: {stats['mean_percentile']*100:.1f}%")
        print(f"  Median Rank: {stats['median_rank']:.0f}")

print("\n" + "="*60)